In [1]:
import requests
import time
import random
import json
import os
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

# =====================
# CONFIGURATION
# =====================
API_KEY = "9706027F573365120F82AC2AA09940A0" #my key
SUPERVISOR_ID = "76561198064675174"

BASE_URL = "https://api.steampowered.com"
RATE_LIMIT = 1.2  # seconds between API calls
MAX_USERS = 100000  
CHECKPOINT_INTERVAL = 500
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# =====================
# LOAD CHECKPOINT IF EXISTS
# =====================
if os.path.exists("checkpoint/visited.json"):
    tqdm.write("Loading checkpoint...")
    with open("checkpoint/visited.json") as f:
        visited = set(json.load(f))
    with open("checkpoint/discovered.json") as f:
        discovered = set(json.load(f))
    users_df = pd.read_csv("checkpoint/users.csv")
    users = {u["steamid"]: u for u in users_df.to_dict("records")}
    edges_df = pd.read_csv("checkpoint/friends.csv")
    edges = [(e["user_steamid"], e["friend_steamid"]) for e in edges_df.to_dict("records")]
    games_df = pd.read_csv("checkpoint/user_games.csv")
    user_games = [(g["steamid"], g["appid"], g["playtime_minutes"]) for g in games_df.to_dict("records")]
    # Rebuild game_counts
    game_counts = defaultdict(int)
    for g in user_games:
        game_counts[g[1]] += 1
else:
    visited = set()
    discovered = set([SUPERVISOR_ID])
    users = {}
    edges = []
    user_games = []
    game_counts = defaultdict(int)

# Ensure checkpoint directory exists
os.makedirs("checkpoint", exist_ok=True)

# =====================
# API FUNCTIONS
# =====================
def get_player_summary(steamid):
    url = f"{BASE_URL}/ISteamUser/GetPlayerSummaries/v2/"
    params = {"key": API_KEY, "steamids": steamid}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return None
    players = r.json().get("response", {}).get("players", [])
    return players[0] if players else None

def get_friends(steamid):
    url = f"{BASE_URL}/ISteamUser/GetFriendList/v1/"
    params = {"key": API_KEY, "steamid": steamid}
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return []
    return [f["steamid"] for f in r.json().get("friendslist", {}).get("friends", [])]

def get_games(steamid):
    url = f"{BASE_URL}/IPlayerService/GetOwnedGames/v1/"
    params = {
        "key": API_KEY,
        "steamid": steamid,
        "include_appinfo": False
    }
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return None
    return r.json().get("response", {}).get("games")

# =====================
# CHECKPOINT FUNCTION
# =====================
def save_checkpoint():
    with open("checkpoint/visited.json", "w") as f:
        json.dump(list(visited), f)
    with open("checkpoint/discovered.json", "w") as f:
        json.dump(list(discovered), f)
    pd.DataFrame(users.values()).to_csv("checkpoint/users.csv", index=False)
    edges_df = pd.DataFrame(edges, columns=["user_steamid", "friend_steamid"])
    edges_df.to_csv("checkpoint/friends.csv", index=False)
    games_df = pd.DataFrame(user_games, columns=["steamid", "appid", "playtime_minutes"])
    games_df.to_csv("checkpoint/user_games.csv", index=False)

# =====================
# MAIN CRAWLER
# =====================
progress = tqdm(total=MAX_USERS, desc="Crawling users", unit="users", ncols=100)
progress.n = len(visited)
progress.last_print_n = len(visited)
progress.refresh()

while len(visited) < MAX_USERS and discovered:
    # Pick random unvisited user
    steamid = random.choice(tuple(discovered - visited))

    # Get user summary
    summary = get_player_summary(steamid)
    time.sleep(RATE_LIMIT)

    # Mark as visited and update progress
    visited.add(steamid)
    progress.update(1)
    progress.refresh()

    if not summary:
        continue

    visibility = summary.get("communityvisibilitystate", 1)

    # Store user info
    users[steamid] = {
        "steamid": steamid,
        "public": visibility == 3,
        "country": summary.get("loccountrycode"),
        "account_created": summary.get("timecreated"),
        "last_logoff": summary.get("lastlogoff")
    }

    # PRIVATE USER → do not expand
    if visibility != 3:
        continue

    # PUBLIC USER → expand friends
    friends = get_friends(steamid)
    time.sleep(RATE_LIMIT)
    for f in friends:
        edges.append((steamid, f))
        discovered.add(f)

    # Get games
    games = get_games(steamid)
    time.sleep(RATE_LIMIT)

    if games:
        for g in games:
            appid = g["appid"]
            playtime = g.get("playtime_forever", 0)
            user_games.append((steamid, appid, playtime))
            game_counts[appid] += 1

    # Periodic checkpoint
    if len(visited) % CHECKPOINT_INTERVAL == 0:
        save_checkpoint()

progress.close()
save_checkpoint()
tqdm.write("Done :D")

# =====================
# FINAL GAME DATA
# =====================
all_games = sorted(game_counts.items(), key=lambda x: x[1], reverse=True)
games_df = pd.DataFrame(all_games, columns=["appid", "user_count"])
games_df["appid"] = games_df["appid"].astype(str)
games_df.to_csv("games.csv", index=False)

Crawling users:   0%|                                     | 5/100000 [00:19<118:32:07,  4.27s/users]

KeyboardInterrupt: 

In [11]:
import requests
import pandas as pd
import time
import os
from tqdm import tqdm
import json

# =====================
# CONFIGURATION
# =====================
GAMES_CSV = "games.csv"  # CSV from your crawler
CHECKPOINT_DIR = "checkpoint_games"
RATE_LIMIT = 0.5  # seconds between API calls
CHECKPOINT_INTERVAL = 50  # save every 50 games

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# =====================
# LOAD GAMES
# =====================
games_df = pd.read_csv(GAMES_CSV)
# Only unique appids
games_df = games_df.drop_duplicates(subset="appid")

# =====================
# LOAD CHECKPOINT IF EXISTS
# =====================
checkpoint_file = os.path.join(CHECKPOINT_DIR, "enriched_games.csv")
visited_file = os.path.join(CHECKPOINT_DIR, "visited_games.json")

# Initialize
enriched_df = pd.DataFrame(columns=["appid", "name", "genres", "developer", "publisher",
                                    "platforms", "release_date", "user_count"])
visited_appids = set()

# Load existing CSV if exists
if os.path.exists(checkpoint_file):
    enriched_df = pd.read_csv(checkpoint_file, sep=";", quoting=1)

# Load visited_appids safely
if os.path.exists(visited_file) and os.path.getsize(visited_file) > 0:
    try:
        with open(visited_file) as f:
            visited_appids = set(int(a) for a in json.load(f))
        tqdm.write(f"Resuming from checkpoint, {len(visited_appids)} games already processed")
    except Exception:
        tqdm.write("Warning: visited_games.json corrupted, starting fresh")

# =====================
# MAIN LOOP
# =====================
for idx, row in tqdm(games_df.iterrows(), total=len(games_df),
                      desc="Fetching game metadata", ncols=100):

    appid = int(row["appid"])  # ensure it's a plain int
    user_count = row["user_count"]

    if appid in visited_appids:
        continue  # skip already fetched games

    url = f"https://store.steampowered.com/api/appdetails?appids={appid}"

    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        if not data[str(appid)]["success"]:
            continue
        info = data[str(appid)]["data"]

        name = info.get("name")
        genres = ";".join([g["description"] for g in info.get("genres", [])]) if info.get("genres") else ""
        developer = ";".join(info.get("developers", [])) if info.get("developers") else ""
        publisher = ";".join(info.get("publishers", [])) if info.get("publishers") else ""
        platforms = ";".join([k for k,v in info.get("platforms", {}).items() if v]) if info.get("platforms") else ""
        release_date = info.get("release_date", {}).get("date", "")

        # Replace tabs with semicolons to avoid CSV issues
        for col in ["genres", "developer", "publisher", "platforms"]:
            val = locals()[col]
            locals()[col] = val.replace("\t", ";") if val else val

        enriched_df = pd.concat([enriched_df, pd.DataFrame([{
            "appid": appid,
            "name": name,
            "genres": genres,
            "developer": developer,
            "publisher": publisher,
            "platforms": platforms,
            "release_date": release_date,
            "user_count": user_count
        }])], ignore_index=True)

        visited_appids.add(appid)

    except Exception as e:
        print(f"Error fetching {appid}: {e}")

    # Save checkpoint periodically
    if len(visited_appids) % CHECKPOINT_INTERVAL == 0:
        enriched_df.to_csv(checkpoint_file, sep=";", index=False, quoting=1)
        with open(visited_file, "w") as f:
            json.dump([int(a) for a in visited_appids], f)

    time.sleep(RATE_LIMIT)

# Final save
enriched_df.to_csv(checkpoint_file, sep=";", index=False, quoting=1)
with open(visited_file, "w") as f:
    json.dump([int(a) for a in visited_appids], f)

tqdm.write("Finished!")

Resuming from checkpoint, 50 games already processed


Fetching game metadata:   2%|▊                                    | 71/3213 [00:17<12:51,  4.07it/s]


KeyboardInterrupt: 